# Reinforcement Learning для LLM

> **Тема курса:** как идея обучения с подкреплением проникла в обучение языковых моделей — от классического RL до RLHF, DPO и reasoning-моделей уровня o1/R1.
>
> **Логика главы:** сначала освежаем базовый RL (чтобы термины не пугали), затем смотрим, *зачем* RL вообще понадобился для LLM, и прослеживаем эволюцию подходов до 2025–2026 гг.

---

## 0. Зачем эта тема вообще существует

Предобучение (next-token prediction) учит модель *продолжать текст похоже на интернет*. Но «похоже на интернет» ≠ «полезно, безопасно и правдиво для пользователя». Возникает зазор между **целью обучения** (правдоподобие) и **целью использования** (полезность). RL — это инструмент, который позволяет оптимизировать модель напрямую под трудноформализуемую цель, заданную не функцией потерь, а **сигналом награды** (человеком, другой моделью или проверяемым правилом).

Ключевая мысль главы: *RL для LLM — это не про игры и роботов, а про оптимизацию под цель, которую нельзя записать как простой лосс.*

---

## Часть I. Базовый RL (review)

### 1.1 Постановка задачи

RL описывает агента, который взаимодействует со **средой** во времени. Формализм — **марковский процесс принятия решений (MDP)**:

- **State** $s$ — состояние среды.
- **Action** $a$ — действие агента.
- **Reward** $r$ — скалярный сигнал, насколько хорошо.
- **Policy** $\pi(a \mid s)$ — стратегия: распределение действий при данном состоянии.
- **Transition** $P(s' \mid s, a)$ — как среда меняется.

Цель агента — максимизировать **ожидаемую суммарную (дисконтированную) награду**:

$$J(\pi) = \mathbb{E}_{\pi}\left[\sum_{t} \gamma^t r_t\right]$$

где $\gamma \in [0,1)$ — фактор дисконтирования (насколько важно будущее).

> **Связь с LLM (запомнить сразу):** состояние = текущий префикс текста (промпт + сгенерированное), действие = выбор следующего токена, политика = сама языковая модель, награда = насколько хорош итоговый ответ. Генерация ответа = один эпизод (траектория).

### 1.2 Ценностные функции

Чтобы понимать «насколько хороша ситуация», вводят:

- **State-value** $V^\pi(s)$ — ожидаемая награда из состояния $s$ при политике $\pi$.
- **Action-value** $Q^\pi(s,a)$ — то же, но если сначала сделать действие $a$.
- **Advantage** $A(s,a) = Q(s,a) - V(s)$ — *насколько действие лучше «среднего» в этом состоянии*. Это центральное понятие для всего, что будет дальше.

### 1.3 Два больших семейства методов

| Семейство | Идея | Примеры |
|---|---|---|
| **Value-based** | Учим $Q$, политика = «бери действие с макс. $Q$» | Q-learning, DQN |
| **Policy-based / Policy-gradient** | Напрямую оптимизируем параметры политики по градиенту награды | REINFORCE, A2C, **PPO** |
| **Actor-Critic** | Гибрид: «актёр» (политика) + «критик» (оценка ценности) | A2C, PPO |

Для LLM почти всё строится на **policy-gradient / actor-critic**, потому что пространство действий (словарь токенов) огромно и дискретно, а политика уже есть — это сама модель.

### 1.4 Policy Gradient — сердце всего дальнейшего

Базовая теорема policy-gradient: градиент награды по параметрам политики

$$\nabla_\theta J = \mathbb{E}\left[\nabla_\theta \log \pi_\theta(a\mid s)\, \cdot\, A(s,a)\right]$$

Интуиция в одну фразу: **повышаем вероятность действий, которые дали награду выше ожидаемой, и понижаем для тех, что хуже.** Множитель — advantage.

Проблемы наивного подхода (**REINFORCE**): огромная дисперсия градиента и нестабильность — один большой шаг может «сломать» политику.

### 1.5 PPO — рабочая лошадка

**Proximal Policy Optimization** (Schulman et al., 2017) решает проблему нестабильности: не давать политике меняться слишком резко за один шаг. Делается это через **clipped objective** — обрезание отношения вероятностей новой и старой политики:

$$L^{\text{CLIP}} = \mathbb{E}\Big[\min\big(r_t(\theta) A_t,\; \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon)\, A_t\big)\Big], \quad r_t(\theta) = \frac{\pi_\theta(a_t\mid s_t)}{\pi_{\theta_{\text{old}}}(a_t\mid s_t)}$$

Запомнить нужно не формулу, а **идею**: PPO — это policy-gradient с «ремнём безопасности», который не даёт обновлению уйти далеко от текущей политики. Именно поэтому PPO стал стандартом для LLM — там цена расходящегося обучения очень высока.

---

## Часть II. RL приходит в LLM

### 2.1 Проблема выравнивания (alignment)

После предобучения и даже после supervised fine-tuning (SFT, дообучение на примерах «промпт → хороший ответ») остаётся фундаментальная трудность:

> Мы не умеем записать «хороший ответ» как функцию потерь. «Полезность», «безвредность», «честность», «уместный тон» — это про человеческие предпочтения, а не про токены.

Зато люди легко **сравнивают**: ответ A лучше ответа B. Отсюда главная идея следующего этапа — *превратить сравнения в обучающий сигнал*.

### 2.2 RLHF — Reinforcement Learning from Human Feedback

Каноническая схема (InstructGPT, Ouyang et al., 2022; легла в основу ChatGPT) — **три стадии**:

**Стадия 1 — SFT.** Дообучаем базовую модель на качественных демонстрациях «инструкция → ответ». Получаем разумную стартовую политику.

**Стадия 2 — Reward Model (RM).** Собираем данные предпочтений: на один промпт генерируем несколько ответов, человек ранжирует их. Обучаем отдельную модель-судью предсказывать «человеческое предпочтение» как скаляр. Обычно через loss Брэдли–Терри:

$$L_{\text{RM}} = -\log \sigma\big(r(x, y_w) - r(x, y_l)\big)$$

где $y_w$ — предпочтённый (winner) ответ, $y_l$ — отвергнутый (loser).

**Стадия 3 — RL-оптимизация (обычно PPO).** Теперь модель = политика, reward model = среда, дающая награду. Оптимизируем политику, чтобы максимизировать награду RM. Критически важная деталь — **KL-штраф**:

$$\text{reward} = r_{\text{RM}}(x,y) - \beta \cdot \text{KL}\big[\pi_\theta(\cdot\mid x)\,\|\,\pi_{\text{SFT}}(\cdot\mid x)\big]$$

> **Зачем KL-штраф (очень важно для интуиции).** Без него политика начинает «взламывать» reward model — находит бессмысленные тексты с аномально высокой наградой (**reward hacking**). KL держит модель рядом с разумной SFT-политикой. Это компромисс «максимизируй награду, но не уходи слишком далеко от языка, который имеет смысл».

**Схема RLHF на одной картинке (текстом):**

```
[Base LM] --SFT--> [SFT model] --(сбор предпочтений)--> [Reward Model]
                         |                                     |
                         +-------------- PPO ------------------+
                                          |
                                   [Aligned model]
                         (награда = RM − β·KL до SFT)
```

### 2.3 Почему это сработало и что сломалось

RLHF дал качественный скачок: модели стали следовать инструкциям, отказываться от вредного, держать тон. Но появились **системные проблемы**:

- **Сложность пайплайна.** Нужно держать одновременно 3–4 модели (политика, reference для KL, reward model, иногда critic). Дорого и инженерно хрупко.
- **Reward hacking.** Модель оптимизирует *прокси* (RM), а не реальную цель. RM несовершенна → модель учится её обманывать (например, многословность, угодливость — *sycophancy*).
- **Нестабильность PPO** на текстах: большая дисперсия, чувствительность к гиперпараметрам.

Эти боли определили всю дальнейшую эволюцию: следующие методы по сути отвечают на вопрос *«как получить пользу RLHF, но дешевле и стабильнее?»*

---

## Часть III. Эволюция: как упрощали и улучшали RL для LLM

### 3.1 DPO — выкидываем RL вообще

**Direct Preference Optimization** (Rafailov et al., 2023) — концептуально красивый ход. Математически показано: если награда определяется через предпочтения, то *оптимальную политику можно выразить через саму политику*, и отдельная reward model вместе с PPO становятся не нужны. Предпочтения оптимизируются **напрямую** простым лоссом, похожим на классификацию:

$$L_{\text{DPO}} = -\log \sigma\!\left(\beta \log\frac{\pi_\theta(y_w\mid x)}{\pi_{\text{ref}}(y_w\mid x)} - \beta \log\frac{\pi_\theta(y_l\mid x)}{\pi_{\text{ref}}(y_l\mid x)}\right)$$

> **Идея для запоминания:** DPO превращает «RL по предпочтениям» в обычное обучение с учителем на парах (хороший, плохой) ответ. Никакого сэмплирования, критика и нестабильного PPO — обучение почти как стандартный fine-tuning.

**Цена удобства:** DPO — *offline*-метод, учится на фиксированном датасете предпочтений и не исследует новые ответы во время обучения. PPO (online) генерирует и оценивает свежие сэмплы, что иногда даёт более высокий потолок качества. Это классический trade-off **простота/стабильность (DPO) vs. потенциал онлайн-исследования (PPO)**.

Вокруг DPO выросло целое семейство: **IPO** (борьба с переобучением на предпочтениях), **KTO** (учится на отдельных метках «хорошо/плохо» без пар), **ORPO** (объединяет SFT и предпочтения в один шаг). Общий вектор — *меньше движущихся частей*.

### 3.2 Сдвиг парадигмы: от «нравится людям» к «верно по факту»

До 2024 года RL для LLM = в основном про **выравнивание под предпочтения** (вкус, тон, безопасность). Затем фокус сместился на **reasoning** — математику, код, логику, где есть **проверяемый правильный ответ**.

Это меняет источник награды кардинально:

| | RLHF / DPO | RL для reasoning |
|---|---|---|
| Источник награды | модель-судья на предпочтениях людей | **правило / верификатор** (ответ верный? код прошёл тесты?) |
| Природа сигнала | субъективный, шумный | объективный, дешёвый, масштабируемый |
| Главный риск | reward hacking, sycophancy | reward sparse (только в конце) |

Этот подход называют **RLVR — Reinforcement Learning with Verifiable Rewards**. Награда не предсказывается обученной RM, а *вычисляется*: сверкой с эталонным ответом, запуском кода, проверкой формата. Дёшево, не взламывается обычными трюками и отлично масштабируется.

### 3.3 GRPO — упрощаем сам RL-алгоритм

Group Relative Policy Optimization (GRPO) — алгоритм обучения с подкреплением, специально предназначенный для усиления способностей к рассуждению у больших языковых моделей. Появился в DeepSeekMath (Shao et al., 2024) и прославился в DeepSeek-R1.

Ключевая проблема PPO для LLM: нужен **critic** (отдельная сеть, оценивающая $V(s)$), сопоставимый по размеру с самой моделью — это удваивает память и вычисления. GRPO избавляется от критика так:

1. На один промпт сэмплируем **группу** из $G$ ответов одной политикой.
2. Каждому ответу вычисляем награду (например, верификатором).
3. **Advantage** считаем не через обученный критик, а относительно **среднего по группе**: насколько этот ответ лучше или хуже своих «соседей».

$$A_i = \frac{r_i - \text{mean}(r_1,\dots,r_G)}{\text{std}(r_1,\dots,r_G)}$$

> GRPO отказывается от явной функции ценности, вместо этого используя среднюю награду по группе сэмплированных ответов на один запрос как базу для оценки advantage. Это существенно сокращает вычислительные ресурсы по сравнению с PPO.

Интуиция в одной фразе: **вместо отдельной модели-критика берём «одноклассников» — другие ответы на тот же вопрос — как точку отсчёта.**

### 3.4 DeepSeek-R1 — кульминация идеи

Главный результат, всколыхнувший область в начале 2025: способности к рассуждению можно стимулировать через чистое обучение с подкреплением, опираясь только на корректность финальных предсказаний относительно эталонных ответов, без ограничений на сам процесс рассуждения.

Два важных артефакта:

- **R1-Zero.** Обходит стандартную стадию supervised fine-tuning перед RL. Гипотеза авторов: человеко-заданные паттерны рассуждения могут ограничивать исследование модели, тогда как свободное RL-обучение лучше стимулирует появление новых способностей к рассуждению. Результат: модель самостоятельно выработала разнообразные и сложные виды рассуждающего поведения. Сюда входят самопроверка и рефлексия — без явного обучения этому.
- **R1.** Чтобы убрать практические дефекты R1-Zero (плохая читаемость, смешение языков), используется многостадийный пайплайн, чередующий SFT и RL: небольшой «cold-start» SFT для формата, затем RL на проверяемых наградах, затем rejection sampling + SFT.

> **Почему это веха.** RLHF использовал RL для *вкуса*. R1 показал, что RL может *создавать новую способность* — длинное рассуждение — практически из ничего, лишь под давлением сигнала «ответ верный/неверный». Эмерджентное удлинение цепочки рассуждений под наградой стало одним из самых обсуждаемых явлений года.

### 3.5 Что пошло дальше (2025–2026)

Стандартный GRPO не идеален — он часто сталкивается с коллапсом энтропии, коллапсом награды и нестабильностью обучения из-за отсутствия процессного контроля. Появились уточнения:

- **Dr. GRPO** — устраняет смещения, связанные со стандартным отклонением по сэмплам и зависимостью от длины ответа в стандартном GRPO.
- **GSPO** — переносит оптимизацию на уровень последовательности вместо токенных отношений, повышая стабильность для крупных Mixture-of-Experts моделей.
- **Process reward models (PRM)** — награда не только за финальный ответ, но и за **отдельные шаги** рассуждения. Решает проблему «sparse reward»: сигнал приходит не только в самом конце.
- **Агентный RL** — RL применяют к моделям, использующим инструменты: поиск, код-исполнение, RAG. Награда — за успешное решение задачи с инструментами (Search-R1, RAG-R1 и др.).

---

## Часть IV. Сквозные идеи (то, что спросят на собеседовании)

**1. RL для LLM ≠ классический RL.** Эпизод короткий (одна генерация), среда зачастую «вырождена» (нет реальной динамики state-transition между промптами), главная сложность — в *источнике награды*, а не в долгосрочном планировании.

**2. Эволюция = борьба со сложностью и с reward hacking.** Линия развития: PPO (мощно, но дорого и хрупко) → DPO (выкинули RL, стало просто, но offline) → GRPO (вернули online-RL, но выкинули критик) → RLVR/R1 (сменили источник награды на проверяемый).

**3. Природа награды определяет всё.**
- *Обученная RM на предпочтениях* → выравнивание, но риск взлома и угодливости.
- *Проверяемое правило (RLVR)* → reasoning, дёшево и устойчиво, но только там, где есть «правильный ответ».

**4. KL-регуляризация — неотъемлемая часть.** Почти везде есть якорь к reference-политике (явный в RLHF/DPO, частично и в reasoning-RL). Без него модель уходит в вырожденные решения.

**5. Граница SFT и RL размывается.** Современные пайплайны (R1) — это не «SFT, потом RL», а *чередование* нескольких стадий обоих типов.

### Шпаргалка: сравнение методов

| Метод | Год | Нужна RM? | Нужен critic? | Online? | Источник награды | Главная идея |
|---|---|---|---|---|---|---|
| **PPO (RLHF)** | 2022 | да | да | да | RM на предпочтениях | clipped policy-gradient |
| **DPO** | 2023 | нет | нет | нет | пары предпочтений | RL → классификация на парах |
| **GRPO** | 2024 | нет* | **нет** | да | правило/RM | advantage из группы сэмплов |
| **RLVR / R1** | 2025 | нет | нет | да | **верификатор** | reasoning из чистого RL |

\* В reasoning-варианте RM не нужна — награда вычисляется правилом.

---

## Что я бы добавил от себя (для полноты картины)

- **Reward hacking как сквозная тема.** Это не баг конкретного метода, а фундаментальное свойство оптимизации под прокси (закон Гудхарта: «когда мера становится целью, она перестаёт быть хорошей мерой»). Полезно подать как красную нить через всю лекцию.
- **Constitutional AI / RLAIF.** Логичное продолжение RLHF: feedback даёт не человек, а другая модель по набору принципов («конституции»). Снимает узкое место — дорогую человеческую разметку. Хорошо ложится между разделами 2.2 и 3.1.
- **Inference-time compute.** R1/o1 связали RL не только с обучением, но и с идеей «думать дольше на инференсе» (длинные цепочки рассуждений). Это перекидывает мостик к отдельной теме «test-time scaling», если она у вас будет.
- **Открытый вопрос для дискуссии со студентами:** RL *создаёт* новые способности или лишь *вытягивает* то, что уже заложено предобучением? Споры об этом активны до сих пор и хорошо завершают тему.

---

*Замечание по точности: блок про базовый RL, RLHF и DPO — это устоявшийся материал. Раздел про GRPO/R1 и развитие 2025–2026 опирается на свежие источники (DeepSeek-R1 в Nature, обзоры по post-training) и в этой области всё быстро меняется — перед лекцией стоит свериться с актуальным состоянием.*